# Viz 4 — Robustness Charts

Renders the regression and ML robustness charts: coefficient stability
across sample definitions, R² across samples, and LASSO bootstrap
distribution.

| Chart | Page heading | Artifact |
|---|---|---|
| 18 | Coefficient Stability Across Samples | reg_coef_across_samples.csv |
| 19 | R² Across Samples | reg_r2_across_samples.csv |
| 21 | LASSO Coefficient Stability (Bootstrap) | lasso_bootstrap.csv |

Charts 17 and 20 (rent-decomposition and forest-adjusted scatter) live in
`viz_1_descriptive.ipynb` because they describe the sample rather than
the model.


In [ ]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from _style import (base_layout, save, artifact_path, GRID)

SAMPLE_COLORS = {
    'Main sample': '#4a6fa5',
    'Adj ≥3%': '#c23a3a',
    'Adj ≥2%': '#2e7d4a',
    'Adj ≥1%': '#c9a227',
}


## Chart 18 — Coefficient stability across samples

Five key regressors (HC, Capital Formation, Production Value, and the two
log × log interactions), four sample definitions. Vertically offset so
the four CIs do not overlap.


In [ ]:
across = pd.read_csv(artifact_path('reg_coef_across_samples.csv'))

key_vars = list(across['feature'].drop_duplicates())
samples  = ['Main sample', 'Adj ≥3%', 'Adj ≥2%', 'Adj ≥1%']

fig18 = go.Figure()
fig18.add_vline(x=0, line=dict(color='#ccc', width=1.5))
for si, sn in enumerate(samples):
    sub = across[across['sample'] == sn]
    if len(sub) == 0:
        continue
    sub = sub.set_index('feature').reindex(key_vars).reset_index()
    ypos = [i + (si - 1.5) * 0.12 for i in range(len(key_vars))]
    fig18.add_trace(go.Scatter(
        x=sub['coef'], y=ypos, mode='markers', name=sn,
        marker=dict(size=7, color=SAMPLE_COLORS.get(sn, '#999')),
        error_x=dict(type='data',
                     array=sub['ci_hi'] - sub['coef'],
                     arrayminus=sub['coef'] - sub['ci_lo'],
                     color=SAMPLE_COLORS.get(sn, '#999'), thickness=1.3),
    ))
fig18.update_layout(
    **base_layout(height=460, margin=dict(l=170, r=40, t=20, b=60)),
    xaxis=dict(title='Coefficient', gridcolor=GRID),
    yaxis=dict(tickvals=list(range(len(key_vars))),
               ticktext=[across[across['feature'] == v]['short'].iloc[0]
                         for v in key_vars],
               gridcolor=GRID, tickfont=dict(size=11)),
    legend=dict(x=1.01, y=0.99, font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
)
save(fig18, '18_coef_comparison_across_samples.html')


## Chart 19 — R² across samples


In [ ]:
r2 = pd.read_csv(artifact_path('reg_r2_across_samples.csv'))

fig19 = go.Figure(go.Bar(
    x=r2['sample'], y=r2['r2'],
    marker_color=[SAMPLE_COLORS.get(s, '#999') for s in r2['sample']],
    text=[f'{v:.3f}' for v in r2['r2']], textposition='outside',
    customdata=r2['n'],
    hovertemplate='%{x}<br>R² = %{y:.4f}<br>'
                  'N = %{customdata:,}<extra></extra>',
))
fig19.update_layout(
    **base_layout(height=400, margin=dict(l=70, r=40, t=20, b=60)),
    xaxis=dict(gridcolor=GRID),
    yaxis=dict(title='R²', gridcolor=GRID,
               range=[0, max(r2['r2'])*1.15]),
    showlegend=False,
)
save(fig19, '19_r2_comparison_across_samples.html')


## Chart 21 — LASSO coefficient stability (bootstrap)

Box plot per feature, ranked by median absolute coefficient (excluding
lagged ECI). Each box summarises the distribution of LASSO coefficients
across 200 country-block bootstrap iterations.


In [ ]:
boot = pd.read_csv(artifact_path('lasso_bootstrap.csv'))
boot = boot[boot['short'] != 'Lagged ECI'].copy()

med_abs = (boot.groupby('short')['coef']
                .apply(lambda s: s.abs().median())
                .sort_values(ascending=True))
order = med_abs.index.tolist()
boot['short'] = pd.Categorical(boot['short'], categories=order, ordered=True)

fig21 = go.Figure()
fig21.add_vline(x=0, line=dict(color='#ccc', width=1.5))
fig21.add_trace(go.Box(
    y=boot['short'], x=boot['coef'], orientation='h',
    marker=dict(color='#4a6fa5'),
    line=dict(color='#1a2744'),
    boxpoints=False, name='LASSO',
    hovertemplate='%{y}: %{x:.3f}<extra></extra>',
))
fig21.update_layout(
    **base_layout(height=600, margin=dict(l=170, r=40, t=20, b=60)),
    xaxis=dict(title='LASSO coefficient (200 bootstrap iterations)',
               gridcolor=GRID),
    yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
    showlegend=False,
)
save(fig21, '21_lasso_coefficient_stability.html')


## Done

Robustness charts written to `outputs/`. The bootstrap takes a few minutes
to compute in `prep/3_ml_models.ipynb`; if you re-run the prep notebook
with `BOOTSTRAP_B` set lower (say 50) the chart will still render
correctly, just with wider boxes.
